# Task 3: Energy Consumption Forecasting
### ARIMA, Prophet & XGBoost on UCI Household Power Consumption Dataset

**Intern:** Falak | **Program:** DevelopersHub Corporation - Data Science & Analytics Internship (Advanced Task Set)

**Objective:** Forecast household energy consumption (Global Active Power) using three different forecasting approaches — a classical statistical model (ARIMA), a decomposition-based model (Prophet), and a gradient-boosted machine learning model (XGBoost) — then compare their performance.

**Dataset:** [UCI Individual Household Electric Power Consumption](https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption) — minute-level electricity consumption readings for a single household in Sceaux, France, from December 2006 to November 2010 (~2.07 million rows, 9 columns).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.style.use('seaborn-v0_8-whitegrid') if 'seaborn-v0_8-whitegrid' in plt.style.available else None

pd.set_option('display.max_columns', None)
print('Libraries loaded.')

## 1. Load & Inspect Raw Data

The raw file is semicolon-separated, uses `?` for missing values, and splits date/time into two columns.

In [ ]:
DATA_PATH = 'data/household_power_consumption.txt'

df = pd.read_csv(
    DATA_PATH,
    sep=';',
    na_values=['?'],
    low_memory=False
)

print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Combine Date + Time into a single datetime index
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df = df.drop(columns=['Date', 'Time'])
df = df.set_index('datetime').sort_index()

numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage',
                'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

print('Date range:', df.index.min(), 'to', df.index.max())
print('Missing values per column:')
df.isna().sum()

## 2. Handling Missing Values

About 1.25% of rows have missing readings (documented gaps in the original dataset, e.g. around 28 April 2007). Since this is a minute-level time series, we forward-fill/interpolate rather than drop rows, to preserve continuity.

In [ ]:
missing_pct = df['Global_active_power'].isna().mean() * 100
print(f'Missing Global_active_power: {missing_pct:.2f}%')

df = df.interpolate(method='time')
df = df.bfill()

print('Remaining missing values:', df.isna().sum().sum())

## 3. Resample to Daily Frequency

Minute-level data (2M+ rows) is too granular and noisy for classical forecasting models like ARIMA/Prophet, and would make training/evaluation extremely slow. We resample to **daily average Global Active Power (kW)**, which is the standard framing used for this dataset in forecasting literature, and is the target variable we'll forecast.

In [ ]:
daily = df['Global_active_power'].resample('D').mean().to_frame()
daily.columns = ['Global_active_power']
daily = daily.interpolate()  # in case any full day is missing

print('Daily series shape:', daily.shape)
daily.head()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(daily.index, daily['Global_active_power'], linewidth=0.8, color='#2563eb')
ax.set_title('Daily Average Global Active Power (2006-2010)')
ax.set_xlabel('Date')
ax.set_ylabel('kW')
plt.tight_layout()
plt.savefig('outputs/01_daily_series.png', dpi=110)
plt.show()

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# Monthly seasonality
monthly_avg = daily.groupby(daily.index.month)['Global_active_power'].mean()
axes[0,0].bar(monthly_avg.index, monthly_avg.values, color='#3b82f6')
axes[0,0].set_title('Avg Power by Month (Seasonality)')
axes[0,0].set_xlabel('Month')
axes[0,0].set_ylabel('kW')

# Day of week pattern
dow_avg = daily.groupby(daily.index.dayofweek)['Global_active_power'].mean()
axes[0,1].bar(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], dow_avg.values, color='#10b981')
axes[0,1].set_title('Avg Power by Day of Week')
axes[0,1].set_ylabel('kW')

# Distribution
axes[1,0].hist(daily['Global_active_power'], bins=50, color='#f59e0b', edgecolor='white')
axes[1,0].set_title('Distribution of Daily Avg Power')
axes[1,0].set_xlabel('kW')

# Yearly trend
yearly_avg = daily.groupby(daily.index.year)['Global_active_power'].mean()
axes[1,1].plot(yearly_avg.index, yearly_avg.values, marker='o', color='#ef4444')
axes[1,1].set_title('Yearly Average Trend')
axes[1,1].set_xlabel('Year')
axes[1,1].set_ylabel('kW')

plt.tight_layout()
plt.savefig('outputs/02_eda_overview.png', dpi=110)
plt.show()

**Observations:**
- Consumption is clearly higher in winter months (heating load) and lower in summer.
- Weekends (Sat/Sun) show slightly different usage patterns than weekdays.
- The series shows a mild long-term trend across years alongside strong seasonality — a good candidate for models like Prophet (explicit seasonality) and ARIMA (differencing removes trend).

## 5. Train-Test Split

We hold out the **last 90 days** as the test set for forecasting evaluation, training on everything before that.

In [ ]:
TEST_DAYS = 90

train = daily.iloc[:-TEST_DAYS]
test = daily.iloc[-TEST_DAYS:]

print(f'Train: {train.index.min().date()} to {train.index.max().date()} ({len(train)} days)')
print(f'Test:  {test.index.min().date()} to {test.index.max().date()} ({len(test)} days)')

In [ ]:
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{name:10s} | MAE: {mae:.4f}  RMSE: {rmse:.4f}  R2: {r2:.4f}')
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

results = []
import os
os.makedirs('outputs', exist_ok=True)

## 6. Model 1 — ARIMA

Using `statsmodels`. We first check stationarity with the Augmented Dickey-Fuller test, then fit an ARIMA(p,d,q) model on the training series.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

adf_result = adfuller(train['Global_active_power'])
print(f'ADF Statistic: {adf_result[0]:.4f}')
print(f'p-value: {adf_result[1]:.4f}')
print('Series is', 'stationary' if adf_result[1] < 0.05 else 'non-stationary', '(at 5% significance)')

In [ ]:
# ARIMA(p,d,q) - order chosen via light manual tuning; d=1 for trend, weekly seasonality handled implicitly via q
arima_model = ARIMA(train['Global_active_power'], order=(5, 1, 2))
arima_fit = arima_model.fit()

arima_forecast = arima_fit.forecast(steps=TEST_DAYS)
arima_forecast.index = test.index

res_arima = evaluate(test['Global_active_power'], arima_forecast, 'ARIMA')
results.append(res_arima)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(train.index[-120:], train['Global_active_power'].iloc[-120:], label='Train (last 120d)', color='#94a3b8')
ax.plot(test.index, test['Global_active_power'], label='Actual', color='#1e293b', linewidth=1.5)
ax.plot(test.index, arima_forecast, label='ARIMA Forecast', color='#ef4444', linewidth=1.5, linestyle='--')
ax.set_title('ARIMA: Actual vs Forecast')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/03_arima_forecast.png', dpi=110)
plt.show()

## 7. Model 2 — Prophet

Facebook's Prophet handles trend + multiple seasonalities (weekly, yearly) natively.

In [ ]:
from prophet import Prophet

prophet_train = train.reset_index().rename(columns={'datetime': 'ds', 'Global_active_power': 'y'})

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)
prophet_model.fit(prophet_train)

future = prophet_model.make_future_dataframe(periods=TEST_DAYS, freq='D')
prophet_forecast_full = prophet_model.predict(future)
prophet_forecast = prophet_forecast_full.set_index('ds')['yhat'].iloc[-TEST_DAYS:]

res_prophet = evaluate(test['Global_active_power'], prophet_forecast, 'Prophet')
results.append(res_prophet)

In [ ]:
fig = prophet_model.plot_components(prophet_forecast_full)
plt.tight_layout()
plt.savefig('outputs/04_prophet_components.png', dpi=110)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(train.index[-120:], train['Global_active_power'].iloc[-120:], label='Train (last 120d)', color='#94a3b8')
ax.plot(test.index, test['Global_active_power'], label='Actual', color='#1e293b', linewidth=1.5)
ax.plot(test.index, prophet_forecast, label='Prophet Forecast', color='#8b5cf6', linewidth=1.5, linestyle='--')
ax.set_title('Prophet: Actual vs Forecast')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/05_prophet_forecast.png', dpi=110)
plt.show()

## 8. Model 3 — XGBoost

XGBoost needs the time series reframed as a supervised learning problem: we engineer lag features, rolling statistics, and calendar features, then train a gradient-boosted regressor.

In [ ]:
def make_features(data):
    d = data.copy()
    d['dayofweek'] = d.index.dayofweek
    d['month'] = d.index.month
    d['day'] = d.index.day
    d['quarter'] = d.index.quarter
    d['dayofyear'] = d.index.dayofyear
    d['weekofyear'] = d.index.isocalendar().week.astype(int)

    for lag in [1, 2, 3, 7, 14]:
        d[f'lag_{lag}'] = d['Global_active_power'].shift(lag)

    d['rolling_mean_7'] = d['Global_active_power'].shift(1).rolling(7).mean()
    d['rolling_mean_14'] = d['Global_active_power'].shift(1).rolling(14).mean()
    d['rolling_std_7'] = d['Global_active_power'].shift(1).rolling(7).std()
    return d

full_feat = make_features(daily)
full_feat = full_feat.dropna()

feature_cols = [c for c in full_feat.columns if c != 'Global_active_power']

train_feat = full_feat.loc[:train.index[-1]]
test_feat = full_feat.loc[test.index[0]:]

X_train, y_train = train_feat[feature_cols], train_feat['Global_active_power']
X_test, y_test = test_feat[feature_cols], test_feat['Global_active_power']

print('Train features:', X_train.shape, ' Test features:', X_test.shape)

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

xgb_forecast = pd.Series(xgb_model.predict(X_test), index=X_test.index)

res_xgb = evaluate(y_test, xgb_forecast, 'XGBoost')
results.append(res_xgb)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(train.index[-120:], train['Global_active_power'].iloc[-120:], label='Train (last 120d)', color='#94a3b8')
ax.plot(y_test.index, y_test, label='Actual', color='#1e293b', linewidth=1.5)
ax.plot(xgb_forecast.index, xgb_forecast, label='XGBoost Forecast', color='#10b981', linewidth=1.5, linestyle='--')
ax.set_title('XGBoost: Actual vs Forecast')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/06_xgboost_forecast.png', dpi=110)
plt.show()

In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
importances.plot(kind='bar', ax=ax, color='#10b981')
ax.set_title('XGBoost Feature Importance')
plt.tight_layout()
plt.savefig('outputs/07_xgb_feature_importance.png', dpi=110)
plt.show()

## 9. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(test.index, test['Global_active_power'], label='Actual', color='black', linewidth=2)
ax.plot(test.index, arima_forecast, label='ARIMA', linestyle='--', color='#ef4444')
ax.plot(test.index, prophet_forecast, label='Prophet', linestyle='--', color='#8b5cf6')
ax.plot(xgb_forecast.index, xgb_forecast, label='XGBoost', linestyle='--', color='#10b981')
ax.set_title('Model Comparison: Actual vs Forecasts (Test Period)')
ax.set_ylabel('kW')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/08_model_comparison.png', dpi=110)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].bar(results_df['Model'], results_df['MAE'], color=['#10b981','#8b5cf6','#ef4444'])
ax[0].set_title('MAE by Model (lower is better)')
ax[1].bar(results_df['Model'], results_df['RMSE'], color=['#10b981','#8b5cf6','#ef4444'])
ax[1].set_title('RMSE by Model (lower is better)')
plt.tight_layout()
plt.savefig('outputs/09_metrics_comparison.png', dpi=110)
plt.show()

## 10. Conclusion

- **XGBoost** typically performs best on this dataset because it can leverage lag features and rolling statistics directly, capturing short-term autocorrelation that pure statistical models miss.
- **Prophet** does well at capturing yearly/weekly seasonality automatically without manual feature engineering, making it the fastest to a reasonable baseline.
- **ARIMA** is the most sensitive to hyperparameter choice (p,d,q) and struggles more with the yearly seasonal pattern unless extended to SARIMA.

**Next steps (possible extensions):** hyperparameter tuning (auto_arima / grid search for XGBoost), adding weather data as an exogenous regressor, and forecasting at higher resolution (hourly) for shorter horizons.

---
*Prepared by Falak — DevelopersHub Corporation Data Science & Analytics Internship, Advanced Task Set (Task 3).*